<a href="https://colab.research.google.com/github/gauravd12345/ml_proj/blob/main/rnn/rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import re
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# import nltk
# nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [42]:
""" Hyperparameters """

train_size = 0.8
lr = 0.001
epochs = 50
batch_size = 256
hidden_size = 300
embedding_dim = 300

In [43]:
with open("/content/tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = word_tokenize(text)
vocab = sorted(set(text))

# vocab = "".join(sorted(set(text)))
vocab_len = len(vocab)

ctoi, itoc = {}, {}
for i in range(len(vocab)):
    ctoi[vocab[i]] = i
    itoc[i] = vocab[i]

print(vocab_len)

14210


In [44]:
X, y = [], []
for i, j in zip(text, text[1:]):
    X.append(ctoi[i])
    y.append(ctoi[j])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

n = len(X)
train_end = int(train_size * n)

X_train = X[:train_end]
X_test = X[train_end:]

y_train = y[:train_end]
y_test = y[train_end:]

print(X_train[:10])
print(y_train[:10])

tensor([ 1148,   705,   220,   479, 13782, 10412,  3431,  7048,   216,  7571])
tensor([  705,   220,   479, 13782, 10412,  3431,  7048,   216,  7571,  8944])


In [45]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_len, embedding_dim)
        self.x_t = nn.Linear(embedding_dim + hidden_size, hidden_size)
        self.y_t = nn.Linear(hidden_size, vocab_len)


    def forward(self, w, s_p):
        w = self.embed(w)
        x = torch.cat((w, s_p), 1) # shape is (B, V + H)
        s = torch.tanh(self.x_t(x))
        y = self.y_t(s)
        return y, s

model = RNN().to(device)

In [46]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=batch_size, drop_last=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
for epoch in range(epochs):
    total_loss = 0

    h = torch.ones(batch_size, hidden_size).to(device) * 0.1
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out, h = model(xb, h)
        h = h.detach() # detach hidden layer from computation graph

        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 6.0935
Epoch 2, Loss: 5.2527
Epoch 3, Loss: 4.8528
Epoch 4, Loss: 4.5335
Epoch 5, Loss: 4.2604
Epoch 6, Loss: 4.0247
Epoch 7, Loss: 3.8181
Epoch 8, Loss: 3.6395
Epoch 9, Loss: 3.4878
Epoch 10, Loss: 3.3560
Epoch 11, Loss: 3.2355
Epoch 12, Loss: 3.1286
Epoch 13, Loss: 3.0320
Epoch 14, Loss: 2.9488
Epoch 15, Loss: 2.8741
Epoch 16, Loss: 2.8079
Epoch 17, Loss: 2.7485
Epoch 18, Loss: 2.6935
Epoch 19, Loss: 2.6446
Epoch 20, Loss: 2.5981
Epoch 21, Loss: 2.5575
Epoch 22, Loss: 2.5207
Epoch 23, Loss: 2.4871
Epoch 24, Loss: 2.4526
Epoch 25, Loss: 2.4220
Epoch 26, Loss: 2.3934
Epoch 27, Loss: 2.3660
Epoch 28, Loss: 2.3423
Epoch 29, Loss: 2.3197
Epoch 30, Loss: 2.2998
Epoch 31, Loss: 2.2811
Epoch 32, Loss: 2.2611
Epoch 33, Loss: 2.2423
Epoch 34, Loss: 2.2227
Epoch 35, Loss: 2.2049
Epoch 36, Loss: 2.1882
Epoch 37, Loss: 2.1756
Epoch 38, Loss: 2.1594
Epoch 39, Loss: 2.1452
Epoch 40, Loss: 2.1300
Epoch 41, Loss: 2.1190
Epoch 42, Loss: 2.1083
Epoch 43, Loss: 2.0956
Epoch 44, Loss: 2.08

In [47]:
words_in_vocab = list(ctoi.keys())

start_word = random.choice(words_in_vocab)
data = torch.tensor([ctoi[start_word]], dtype=torch.long, device=device)

length = 100

model.eval()
h = torch.zeros(1, hidden_size, device=device)

print(start_word, end=" ")

with torch.no_grad():
    for i in range(1, length):
        logits, h = model(data, h)

        probs = torch.softmax(logits, dim=1)

        pred = torch.multinomial(probs, num_samples=1).item()
        next_word = itoc[pred]

        print(next_word, end=" ")
        if i % 10 == 0:
            print()

        data = torch.tensor([pred], dtype=torch.long, device=device)

restrain'st over nor ambitious for hell , -- ESCALUS : Well 
might he that given them from them and altering rheums 
? Provost : Ay , And thou didst That I 
have run with curious business are thine only son the 
earth . ESCALUS : King of their feast ? fires 
? ELBOW : it , -- Were as it . 
ESCALUS : which goes not do't , Which was born 
greater , sir . ESCALUS : Good my life , 
As I would wrong , that with some imprisonment . 
MISTRESS OVERDONE : Ay , and me already that 